In [1]:
import numpy as np
from tqdm import tqdm

In [32]:
from data_utils import load_data_all
import torch
"""
dataset = 'gene'
D = np.load('/home2/zzs/dataset/gene/all22.npy')
M = D
print(M.shape)
d1, d2 = M.shape
print(M.shape)
r = 5
"""
"""
dataset = 'netflix'
D = torch.load('/home2/zzs/dataset/netflix/netflix/use_data.pt')
D = D.numpy()
#s_r = 10000
#M=D[:s_r]
M = D
d1, d2 = M.shape
print(M.shape)
r = 5
"""
"""
dataset = 'ml-1m'
D = load_data_all(dataset, s=200)

print(f"original user: {D.shape[0]} item: {D.shape[1]}")
D = D.numpy()
r = 5
"""

"""
dataset = './data/syn/15000.npy'
M = np.load(dataset)
D = M
d1, d2 = M.shape
r = 5
"""

dataset = 'yahoo_music'
D = load_data_all(dataset, s=300)
D = D.numpy()
M = D
d1, d2 = M.shape
r = 5
print(M)


[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [68]:
def compute_matrix_adjustment(V):
    # 计算 V^T * V
    VTV = np.dot(V.T, V)
    
    # 特征值分解 V^T * V
    eigvals, eigvecs = np.linalg.eigh(VTV)
    
    # 计算 (V^T V)^-1/2
    # 我们只取正特征值的逆平方根
    inv_sqrt_eigvals = np.diag([1/np.sqrt(val) if val > 0 else 0 for val in eigvals])
    
    # 重构 (V^T V)^-1/2
    VTV_inv_sqrt = eigvecs @ inv_sqrt_eigvals @ eigvecs.T
    
    # 计算 V * (V^T V)^-1/2
    result = V @ VTV_inv_sqrt
    
    return result

def compute_V_tilde(V_tilde):
    """
    使用 SVD 计算矩阵的平方根。
    
    参数：
    V_tilde -- 输入矩阵，形状为 (n, r)
    
    返回：
    计算后的矩阵 V
    """
    # 计算 V_tilde^T V_tilde
    VtV = V_tilde.T @ V_tilde
    
    # 对 VtV 进行 SVD 分解
    U, S, VT = np.linalg.svd(VtV)
    
    # 计算奇异值的平方根的逆
    S_inv_sqrt = np.diag(1.0 / np.sqrt(S))
    
    # 计算 V
    V = V_tilde @ U @ S_inv_sqrt @ VT
    
    return V

def matrix_negative_half_power(matrix):
    # 特征值分解
    eigvals, eigvecs = np.linalg.eig(matrix)
    
    # 修改特征值：负特征值设为零，正特征值取-1/2次幂
    eigvals[eigvals < 0] = 0
    eigvals = eigvals ** (-1/2)

    matrix_neg_half = eigvecs @ np.diag(eigvals) @ np.linalg.inv(eigvecs)
    
    return matrix_neg_half

def generate_symmetric_gaussian_noise(r, Gamma_u, sigma):
    # 生成随机矩阵
    A = np.random.randn(r, r)
    
    # 使其对称
    A = (A + A.T) / 2
    
    # 调整标准差
    G_j = A * (Gamma_u**4 * sigma)
    
    return G_j

def sample_Omega(Omega, k):
    n, m = Omega.shape
    sampled_indices = []
    
    for i in range(n):
        # 获取当前行的非零列索引
        non_zero_indices = np.where(Omega[i] != 0)[0]
        
        # 如果非零列索引数量少于 k，则全选，否则随机选择 k 个
        if len(non_zero_indices) <= k:
            selected_indices = non_zero_indices
        else:
            selected_indices = np.random.choice(non_zero_indices, k, replace=False)
        
        sampled_indices.append(selected_indices)
    
    return sampled_indices

def project_to_psd_cone(A):
    # 对矩阵 A 进行特征值分解
    eigvals, eigvecs = np.linalg.eigh(A)
    
    # 将所有负的特征值设为零
    eigvals[eigvals < 0] = 0
    
    # 重新构造正半定矩阵
    A_psd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    
    return A_psd

def clip_matrix(M, Gamma_M):
    M = M * max(1, 1/np.linalg.norm(M))
    return M
    #return np.clip(M, -Gamma_M, Gamma_M)

def P_omega(M, Omega):
    P = np.zeros_like(M)
    P[Omega] = M[Omega]
    return P

def Aitem(U, Omega, P_Omega_M, k, lambda_, Gamma_u, Gamma_M):
    n, r = U.shape
    m = P_Omega_M.shape[1]
    V = np.random.rand(m, r)
    Omega_prime = sample_Omega(Omega.T, k)
    
    for j in range(m):
        #G_j = np.random.normal(0, Gamma_u**4 * sigma**2, (r, r))
        G_j = generate_symmetric_gaussian_noise(r, Gamma_u, sigma)
        g_j = np.random.normal(0, Gamma_u**2 * Gamma_M**2 * sigma**2, r)
        X_j = lambda_ * np.eye(r) + sum([np.outer(U[i],U[i]) for i in Omega_prime[j]]) + G_j
        V_j = np.linalg.pinv(project_to_psd_cone(X_j)) @ (sum([P_Omega_M[i, j] * U[i] for i in Omega_prime[j]])+g_j)
        #V_j = project_to_psd_cone(np.linalg.pinv(X_j) @ (sum([P_Omega_M[i, j] * U[i] for i in Omega_prime[j]])+g_j))
        V[j] = V_j
    
    V_tilde = np.vstack(V)
    #VV = matrix_negative_half_power(np.linalg.pinv(V_tilde.T @ V_tilde))
    VV = np.linalg.pinv(project_to_psd_cone(V_tilde.T @ V_tilde))
    #print(VV)
    #print(np.linalg.pinv(VV))
    V = compute_V_tilde(V_tilde)
    return compute_matrix_adjustment(V_tilde)
    #return V
    #return V_tilde @ np.linalg.pinv(V_tilde.T @ V_tilde) ** 0.5
    #return V_tilde @ VV ** 0.5

def Auser(V, Omega_i, P_Omega_M, T, lambda_, Gamma_u):
    def update_u(V, Omega_i_prime, M, lambda_):
        r = V.shape[1]
        I = np.eye(r)
        
        # 计算 lambda I + sum(V_j V_j^T)
        A = lambda_ * I
        for j in Omega_i_prime:
            V_j = V[j]
            A += np.outer(V_j, V_j)
        
        # 计算 sum(M_ij V_j)
        b = np.zeros(r)
        for j in Omega_i_prime:
            M_ij = M[j]
            V_j = V[j]
            b += M_ij * V_j
        
        # 计算 u
        u = np.linalg.solve(A, b)
        
        return u
    m, r = V.shape
    U = np.zeros((n, r))

    nonzero_indices = [np.where(Omega_i[i] != 0)[0] for i in range(n)]
    Omega_prime_i = [np.random.choice(nonzero_indices[i], size=max(1, len(nonzero_indices[i]) // T), replace=False) for i in range(n)]

    for i in range(n):
        #print((sum([ V[j] for j in Omega_prime_i[i]])).shape)
        X_i = lambda_ * np.eye(r) + sum(np.outer(V[j], V[j]) for j in Omega_prime_i[i])
        u_i = np.linalg.pinv(X_i) @ sum([P_Omega_M[i, j] * V[j] for j in Omega_prime_i[i]])
        #U[i] = np.clip(u_i, -Gamma_u, Gamma_u)
        U[i] = clip_matrix(u_i, Gamma_u)

    return U

def DPALS(P_Omega_M, Omega, sigma, Gamma_u, Gamma_M, T, lambda_, r, k, V0):
    n, m = P_Omega_M.shape
    V_t = V0
    U_t = np.random.rand(n, r)
    
    for t in tqdm(range(T)):
        U_t = Auser(V_t, Omega, P_Omega_M, T, lambda_, Gamma_u)
        V_t = Aitem(U_t, Omega, P_Omega_M, k, lambda_, Gamma_u, Gamma_M)
    
    return U_t.T, V_t.T

# 示例初始化
runs = 10
rmse_list = []
G_list = []
"""
min_ret = 1000
for sigma in [0.1, 0.01]:
    for Gamma_u in [2,3,4]:
        for Gamma_M in [2,3,4]:
            for T in [5, 10 ,20]:
"""
for run in range(runs):
    p=0.75
    n, m = D.shape
    # 示例初始化
    Omega = np.random.rand(n, m) <= p 
    Omega = Omega * (D!=0)
    non_zero_rows = np.any(D * Omega != 0, axis=1)
    M = D[non_zero_rows]
    Omega = Omega[non_zero_rows]
    n, m = M.shape
    
    sigma = 0.1
    Gamma_u = 4
    Gamma_M = 2
    T = 20
    
    lambda_ = 0.01
    k = 40
    V0 = np.random.rand(m, r)  # 初始 V

    P_Omega_M = clip_matrix(M*Omega, Gamma_M)
    U_final, V_final = DPALS(P_Omega_M, Omega, sigma, Gamma_u, Gamma_M, T, lambda_, r, k, V0)
    #mask = M>0.1
    mask = M!=0
    num = np.sum(mask)
    #print(num)
    def P_omega_test(X):
        return X * (~Omega * mask)
    def P(X):
        return X * mask
    X_estimation = U_final.T @ V_final
    rand_mat = np.random.rand(n,m)
    rmse_list.append((np.linalg.norm(P_omega_test(M - X_estimation)))/np.sqrt(np.sum((~Omega * mask))))

    MTM = M.T @ M
    XTX = X_estimation.T @ X_estimation
    G_list.append(np.linalg.norm(XTX-MTM) / np.linalg.norm(MTM))
    #print((np.linalg.norm(P(M - X_estimation)))/np.sqrt(num))
    #print((np.linalg.norm(P_omega_test(M - X_estimation)))/np.sqrt(np.sum((~Omega * mask))))

    """
    if rmse_list[-1] < min_ret:
        min_ret = rmse_list[-1]
        sigma_best = sigma
        Gamma_u_best = Gamma_u
        Gamma_M_best = Gamma_M
        T_best = T
    """
    
#print("Final U^T:\n", U_final)
#print("Final V^T:\n", V_final)
print(np.mean(rmse_list))
print(np.std(rmse_list))
print(np.mean(G_list))
print(np.std(G_list))
"""
print("grid search")
print(min_ret)
print(sigma_best)
print(Gamma_u_best)
print(Gamma_M_best)
print(T_best)
"""

  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 20/20 [00:03<00:00,  5.84it/s]

3.637449694164279
0.13491754220102634
37.45716292020826
12.458403627775164


'\nprint("grid search")\nprint(min_ret)\nprint(sigma_best)\nprint(Gamma_u_best)\nprint(Gamma_M_best)\nprint(T_best)\n'

In [4]:

mask = M>0.1
num = np.sum(mask)
print(num)
def P_omega_test(X):
    return X * (~Omega * mask)
def P(X):
    return X * mask
X_estimation = U_final.T @ V_final
rand_mat = np.random.rand(n,m)
print((np.linalg.norm(P(M - X_estimation)))/np.sqrt(num))
print((np.linalg.norm(P_omega_test(M - X_estimation)))/np.sqrt(np.sum((~Omega * mask))))
print((np.linalg.norm(P_omega_test(M - rand_mat)))/np.sqrt(np.sum((~Omega * mask))))

37955
1.7910437719000578
1.8605341525681367
3.2251291829667794


In [5]:
MTM = M.T @ M
VTV = V_final.T @ V_final
print(np.linalg.norm(VTV-MTM) / np.linalg.norm(MTM))

0.9999978060436351


In [6]:
MTM = M.T @ M
XTX = X_estimation.T @ X_estimation
print(np.linalg.norm(XTX-MTM) / np.linalg.norm(MTM))

0.7598464948876701
